# P2 Images 2 — Cleaning, Preprocessing & Image EDA

**MSDS 565 | TeamKAI — KeiTonia, Ashley, India**  
**Notebook 2 of 4 | CRISP-DM: Data Preparation**

Turns the scraped folders into a clean, model-ready, split dataset. Only cleaning, preprocessing and EDA happen here. No training and no saliency maps.

> **How to run this notebook**
> 1. Run the cells through **section 1.1** to load the data and see the duplicate candidates.
> 2. **Clean the class folders by hand** in Finder (see section 1).
> 3. Restart the kernel and run the **whole notebook top to bottom**.

In [1]:
import json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from sklearn.model_selection import train_test_split

# Same local folder Notebook 1 scraped into (outside the repo).
DATA_ROOT = Path.home() / "MSDS565_data"
RAW_DIR = DATA_ROOT / "P2_images"          # one sub-folder per class
OUT_DIR = DATA_ROOT / "P2_processed"       # cleaned, resized, split arrays go here
OUT_DIR.mkdir(parents=True, exist_ok=True)
IMG_SIZE = 224
SEED = 42

# Read the manifest and the scraped folders fresh. Nothing is carried over from Notebook 1.
manifest = pd.read_csv("data/P2_manifest.csv")

# The manifest stores bare filenames (e.g. cumulus_0000.jpg); each lives in its class folder.
manifest["relpath"] = manifest["label"] + "/" + manifest["filename"]

class_names = sorted(manifest["label"].unique())
print(class_names)
print(len(manifest), "rows in the manifest")

['altocumulus', 'cirrus', 'cumulonimbus', 'cumulus', 'stratus']
2000 rows in the manifest


**In simple terms: loading**
- Reads the spreadsheet from Notebook 1 and points at the picture folders on this computer. Nothing is reused from Notebook 1's memory.
- `relpath` is just "folder/filename" (like `cumulus/cumulus_0000.jpg`) so the code can find each picture. It is removed again before the manifest is saved.
- **Output:** the 5 class names (alphabetical) and how many rows the manifest has.

## 1. Cleaning summary (done by hand)

We opened every class folder in Finder and removed junk by hand. No automated method reliably tells that a search for "cumulonimbus" returned a poster, so the removals are manual. The code in 1.1 only **points out** likely duplicates to look at; it deletes nothing.

**What we removed:**
- **Wrong subjects:** drawings and paintings, memes, product collages, infographics and diagrams, photos of the word, weather-app screenshots, satellite or map images, ground-level fog with no clouds, and photos with no visible sky.
- **Wrong cloud type:** photos clearly showing a different cloud than the folder's label.
- **Duplicates:** overlapping queries return the same image more than once, sometimes in two different classes. For a same-class pair we kept one copy. For a cross-class pair we kept the copy in the correct class and deleted the other (or both, if neither label was clearly right).

**✏️ To fill in after cleaning:** one sentence on what made up most of the removals in each class. The per-class counts are in the table in 1.2.

### 1.1 Duplicate candidates to review

Each picture is shrunk to a tiny 9×8 grayscale "fingerprint" (a difference hash). Two pictures whose fingerprints differ in **4 or fewer** of 64 spots are listed as likely duplicates. This flags candidates only. **Before cleaning**, use the list and the side-by-side view to find pairs in Finder. **After cleaning**, re-running it should show few or none left.

In [2]:
def dhash(path, size=8):
    with Image.open(path) as im:
        a = np.asarray(im.convert("L").resize((size + 1, size)), dtype=np.int16)
    return (a[:, 1:] > a[:, :-1]).flatten()

def duplicate_candidates(max_dist=4):
    present = manifest[manifest["relpath"].map(lambda f: (RAW_DIR / f).exists())].reset_index(drop=True)
    h = np.stack([dhash(RAW_DIR / f) for f in present["relpath"]]).astype("float32")
    agree = h @ h.T + (1 - h) @ (1 - h).T
    dist = h.shape[1] - agree
    i, j = np.where(np.triu(dist <= max_dist, k=1))
    return [(present.relpath[a], present.relpath[b], int(dist[a, b])) for a, b in zip(i, j)]

pairs = duplicate_candidates()
cross = [p for p in pairs if p[0].split("/")[0] != p[1].split("/")[0]]
print(f"{len(pairs)} near-duplicate pairs remaining ({len(cross)} across different classes)")

# Full list saved for searching in Finder (outside the repo).
pairs_df = pd.DataFrame(pairs, columns=["image_a", "image_b", "difference"])
pairs_df["cross_class"] = pairs_df.image_a.str.split("/").str[0] != pairs_df.image_b.str.split("/").str[0]
pairs_df.to_csv(OUT_DIR / "duplicate_pairs_to_review.csv", index=False)
print("Full list saved to", OUT_DIR / "duplicate_pairs_to_review.csv")
pairs_df.head(10)

ValueError: need at least one array to stack

In [ ]:
# Side-by-side view of duplicate candidates. Change START to page through them.
START, N_SHOW = 0, 8
show = pairs_df.iloc[START:START + N_SHOW]
if len(show):
    fig, axes = plt.subplots(len(show), 2, figsize=(7, 3 * len(show)), squeeze=False)
    for r, (a, b) in enumerate(zip(show.image_a, show.image_b)):
        for k, f in enumerate((a, b)):
            axes[r, k].imshow(Image.open(RAW_DIR / f).convert("RGB"))
            axes[r, k].set_title(f, fontsize=9); axes[r, k].axis("off")
    plt.tight_layout(); plt.show()
else:
    print("No duplicate candidates to show.")

**In simple terms: finding duplicates**
- The first cell gives every picture a tiny fingerprint and lists pairs whose fingerprints nearly match. The full list is saved as `duplicate_pairs_to_review.csv`.
- The second cell shows those pairs **side by side**, with their file names, so you can find them in Finder. Change `START` to 8, 16, … to see more.
- **Pairs across two classes matter most:** the same photo with two labels confuses the model, and if one copy lands in train and the other in test, the test score is inflated.
- Some flagged pairs are just two plain blue skies that look alike. Only delete real copies.

### 1.2 Accounting: every removal per class
A manifest row that was readable but whose file is no longer on disk was removed during cleaning.

In [ ]:
# Account for every removal: manifest rows that were readable but no longer exist on disk.
manifest["kept"] = manifest["relpath"].map(lambda f: (RAW_DIR / f).exists())

acct = (manifest[manifest["readable"]]
        .groupby("label")
        .agg(before_cleaning=("filename", "size"), after_cleaning=("kept", "sum")))
acct["removed"] = acct["before_cleaning"] - acct["after_cleaning"]
acct["clears_200"] = acct["after_cleaning"] >= 200
print(f"Removed {acct.removed.sum()} of {acct.before_cleaning.sum()} images in total")
acct

In [ ]:
kept = manifest[manifest["kept"]].reset_index(drop=True)
assert (kept.groupby("label").size() >= 200).all(), "A class is under 200 usable images. Pick another class."
print("Every class clears the 200-image minimum.")

**In simple terms: the cleaning count**
- `before_cleaning` = pictures from Notebook 1, `after_cleaning` = pictures still in the folder, `removed` = the difference.
- `clears_200 = True` means the class still has at least 200 pictures, which the assignment requires.
- The `assert` line stops the notebook with an error if any class drops below 200.

## 2. Image EDA

### 2.1 Sample grid (8 images per class)

In [ ]:
rng = np.random.default_rng(SEED)
fig, axes = plt.subplots(len(class_names), 8, figsize=(16, 2.1 * len(class_names)))
for r, c in enumerate(class_names):
    files = rng.choice(kept[kept.label == c].relpath.values, 8, replace=False)
    for k, f in enumerate(files):
        axes[r, k].imshow(Image.open(RAW_DIR / f).convert("RGB"))
        axes[r, k].axis("off")
    axes[r, 0].set_title(c, loc="left", fontsize=11)
plt.tight_layout()
plt.show()

**In simple terms: sample grid**
- 8 random cleaned pictures per cloud type, one row each, so the reader can see what the dataset really looks like.
- `SEED = 42` picks the same random pictures every time the notebook runs.

### 2.2 Image dimensions and aspect ratios

In [ ]:
def size_of(f):
    with Image.open(RAW_DIR / f) as im:
        return im.size

sizes = np.array([size_of(f) for f in kept.relpath])
kept["w"], kept["h"] = sizes[:, 0], sizes[:, 1]
kept["aspect"] = kept.w / kept.h

stats = pd.DataFrame({s: kept[s].agg(["min", "max", "median"]) for s in ["w", "h", "aspect"]}).round(2)
print(stats)
print(f"\nWidths range from {int(stats.w['min'])} to {int(stats.w['max'])} px "
      f"(about {stats.w['max'] / stats.w['min']:.0f}x apart); median aspect ratio {stats.aspect['median']}.")

fig, ax = plt.subplots(1, 3, figsize=(14, 3.5))
ax[0].hist(kept.w, bins=40); ax[0].set_title("Width (px)")
ax[1].hist(kept.h, bins=40); ax[1].set_title("Height (px)")
ax[2].hist(kept.aspect, bins=40); ax[2].set_title("Aspect ratio (w/h)")
plt.tight_layout(); plt.show()

**Interpretation.** The table and the sentence under it give the min, max, and median width, height, and aspect ratio. Web images are **wildly inconsistent** in size: widths span many times over, from small web images to full camera photos. The scraper's filters cap the extremes (at least 224 px on each side, aspect ratio 0.5–2.0), and the median aspect ratio is above 1, so most sky photos are **landscape** (wider than tall). Because the model needs one fixed size, the resize choice in 3.2 matters.

**In simple terms**
- Pictures come in all sizes, so we must make them all the same size before training.
- Most are wider than they are tall, which affects how much padding they get in 3.2.

### 2.3 Class support

In [ ]:
support = kept.label.value_counts().reindex(class_names)
print(support)
ratio = support.max() / support.min()
severity = "mild" if ratio < 1.5 else "moderate" if ratio < 3 else "severe"
print(f"imbalance ratio (max/min): {ratio:.2f} -> {severity}")
support.plot.bar(title="Images per class after cleaning", rot=0)
plt.ylabel("count"); plt.show()

**Severity.** The ratio of the largest class to the smallest is printed above, labeled **mild** (under 1.5), **moderate** (1.5–3), or **severe** (over 3). Imbalance is **not fixed here**. Balancing and augmentation happen in Notebook 3, on the training set only, after the split.

**In simple terms**
- Counts how many pictures each cloud type has, and shows it as a bar chart.
- If the bars are close in height, the classes are balanced. We only measure it here; Notebook 3 fixes it if needed.

## 3. Preprocessing decisions

### 3.1 Color mode: everything to 3-channel RGB
Scraped files mix grayscale, RGBA/PNG-with-alpha and palette images. The pretrained models expect exactly 3 channels, so `convert("RGB")` normalizes all of them. A grayscale or alpha image would otherwise break batching or silently give a wrong-shaped array.

### 3.2 Resize strategy: preserve aspect ratio and pad (letterbox) to 224×224
Options considered:
- **Stretch/squash:** keeps all pixels but distorts shape. For clouds, texture and the proportions of heaped or layered structures are the signal, and distortion corrupts them (cumulus towers would look wider).
- **Center-crop:** keeps true proportions but throws away the left/right edges of landscape photos, which can hold the cloud itself (e.g. an anvil top) or the horizon context.
- **Pad (chosen):** keeps proportions and every pixel.

**What padding costs for this subject:** wide skies get gray bars above and below, so the cloud fills a smaller part of the 224×224 input and effective resolution drops. Worse, the bar size follows the original aspect ratio, which could line up with class through how images were searched (a shortcut). We pad with a constant mid-gray (128) and will test for this in Notebook 4 by greying out the border region.

**The same function, size (224×224) and pad value must be used in the Streamlit app.**

In [ ]:
def letterbox(path_or_img, size=IMG_SIZE, pad_value=128):
    """RGB, scale to fit inside size x size preserving aspect ratio, center on a gray canvas. Returns uint8."""
    im = path_or_img if isinstance(path_or_img, Image.Image) else Image.open(path_or_img)
    im = im.convert("RGB")
    scale = size / max(im.size)
    new = (max(1, round(im.width * scale)), max(1, round(im.height * scale)))
    im = im.resize(new, Image.BILINEAR)
    canvas = Image.new("RGB", (size, size), (pad_value,) * 3)
    canvas.paste(im, ((size - new[0]) // 2, (size - new[1]) // 2))
    return np.asarray(canvas, dtype=np.uint8)

label_to_index = {c: i for i, c in enumerate(class_names)}
x_all = np.stack([letterbox(RAW_DIR / f) for f in kept.relpath])   # raw 0-255 uint8
y_all = kept.label.map(label_to_index).values
print(x_all.shape, x_all.dtype, x_all.min(), x_all.max())

**In simple terms: resizing**
- Every picture becomes **224 × 224** in normal color. It is shrunk to fit without stretching, and the empty space is filled with gray bars.
- `label_to_index` turns each cloud name into a number (alphabetical: altocumulus = 0 … stratus = 4), because models work with numbers.
- **Output:** the shape of all pictures together, `(number of pictures, 224, 224, 3)`, stored as values from 0 to 255.

### 3.3 Per-class mean image (EDA, computed on the resized images)

This EDA item comes after 3.2 because the mean must be taken *after* resizing. A mean image that shows a recognizable shape or a consistent colored border is a shortcut waiting to happen. Gray bars at the top and bottom mean padding is leaking aspect-ratio information.

In [ ]:
fig, axes = plt.subplots(1, len(class_names), figsize=(3 * len(class_names), 3.2))
for k, c in enumerate(class_names):
    axes[k].imshow(x_all[y_all == k].mean(axis=0).astype("uint8"))
    axes[k].set_title(c); axes[k].axis("off")
plt.tight_layout(); plt.show()

**What the means show.** ✏️ *To fill in after running:* describe each class's average picture, for example a bright top and darker bottom (sky over landscape), gray bars from padding, or a class-specific color (gray for stratus, deep blue for cirrus, dark base for cumulonimbus). Flag anything to re-check in Notebook 4.

**In simple terms**
- Averages all pictures in a class into one blurry picture.
- If one class's average shows a clear shape or border the others don't, the model might learn that shortcut instead of the cloud.

## 4. Split and export

Stratified 80/20 split, `random_state=42`. **Nothing is balanced or augmented before the split.** Doing either first would leak duplicated or synthetic copies of training images into the test set and make accuracy fiction. Both happen in Notebook 3 on the training set only.

In [ ]:
idx = np.arange(len(y_all))
tr_idx, te_idx = train_test_split(idx, test_size=0.2, stratify=y_all, random_state=SEED)
x_train, y_train = x_all[tr_idx], y_all[tr_idx]
x_test, y_test = x_all[te_idx], y_all[te_idx]
print("train", x_train.shape, np.bincount(y_train), "| test", x_test.shape, np.bincount(y_test))

In [ ]:
# Format: .npy arrays, raw 0-255 uint8, already resized to 224x224x3, in OUT_DIR (outside the repo).
np.save(OUT_DIR / "x_train.npy", x_train); np.save(OUT_DIR / "y_train.npy", y_train)
np.save(OUT_DIR / "x_test.npy", x_test);   np.save(OUT_DIR / "y_test.npy", y_test)

# One label->index mapping used by Notebooks 3 and 4 and the app.
with open(OUT_DIR / "label_map.json", "w") as f:
    json.dump(label_to_index, f, indent=2)
print(label_to_index)

# Update (do not overwrite) the manifest: add `kept` and which split each kept image went to.
split = pd.Series("", index=kept.index)
split.iloc[tr_idx] = "train"; split.iloc[te_idx] = "test"
kept["split"] = split.values
manifest = manifest.drop(columns=["split"], errors="ignore")   # safe to re-run
manifest = manifest.merge(kept[["filename", "split"]], on="filename", how="left")
manifest.drop(columns=["relpath", "w", "h", "aspect"], errors="ignore").to_csv("data/P2_manifest.csv", index=False)
print(manifest.drop(columns=["relpath"]).columns.tolist())

**In simple terms: split and save**
- **80%** of the cleaned pictures go to **train** (to teach the model) and **20%** to **test** (to quiz it later). "Stratified" keeps the same mix of cloud types in both.
- Saves 4 files (`x_train`, `y_train`, `x_test`, `y_test`) plus `label_map.json`, the name-to-number list Notebooks 3, 4 and the app must all use.
- Updates the spreadsheet with `kept` (survived cleaning?) and `split` (train or test). Notebook 1's columns stay as they were.

**Artifacts for later notebooks** (in `~/MSDS565_data/P2_processed/`, outside the repo): `x_train.npy`, `y_train.npy`, `x_test.npy`, `y_test.npy`, `label_map.json`. Arrays are raw 0–255, letterboxed to 224×224×3. `data/P2_manifest.csv` now also has `kept` and `split`.